# Adaptação brasileira — horizonte cinco minutos

Resolução real de **cinco minutos**, horizonte **t+5 minutos**. Os nomes dos arquivos foram mantidos por solicitação do usuário; o conteúdo foi migrado de diário para intradiário.

O alvo e a persistência (t−5 min) precisam ser originalmente medidos. Cada etapa de preenchimento causal usa até 15 minutos; no Brasil a combinação proxy + ffill pode atingir 30 minutos somente no histórico: nunca se interpola usando ponto posterior. Não se concatenam blocos interrompidos. A avaliação principal é diurna (elevação solar > 0); a avaliação 24h é complementar. Zero noturno não deve inflar a precisão.

As entradas principais são potência AC em p.u., hora local e dia do ano em seno/cosseno. O vento continua excluído. ERA5 nativa horária é auxiliar e não entra nos modelos principais. Estatísticas de normalização vêm apenas do treino australiano; alvos de validação/teste não ajustam escalas.


In [1]:
from pathlib import Path
import sys,json
import numpy as np
import pandas as pd
from IPython.display import display,Image,Markdown
RAIZ=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'src/energia_transfer').is_dir())
if str(RAIZ/'src') not in sys.path: sys.path.insert(0,str(RAIZ/'src'))
PASTA=RAIZ/'experimentos/transfer_learning_australia_brasil'
AU=RAIZ/'experimentos/australia_deep_learning'
REPORT=RAIZ/'reports/intradiario'
from energia_transfer.transfer_learning.dados import carregar,particao
_,CONFIG,CONTRATO=carregar(PASTA)
def exibir(t): display(t.round(5))


In [2]:
from energia_transfer.deep_learning.dados import sha256
decisao=json.loads((PASTA/'resultados/decisao_validacao.json').read_text(encoding='utf-8'))
assert decisao['contrato_sha256']==sha256(PASTA/'dados/contrato.json')
ranking=pd.read_csv(PASTA/'resultados/brasil_ranking_validacao.csv').sort_values('RMSE_medio')
exibir(ranking)
print('Escolha pela validação:',decisao['melhor_rede_validacao'])
print('Incluindo persistência:',decisao['melhor_incluindo_persistencia'])
execucoes=pd.read_csv(PASTA/'resultados/brasil_execucoes.csv')
assert execucoes.query("metodo in ['cabeca','sem_ajuste']").encoder_igual_fonte.all()
exibir(execucoes.groupby(['arquitetura','metodo']).parametros_treinaveis.first().reset_index())


Escolha pela validação: {'arquitetura': 'lstm_direta', 'lag': 12, 'metodo': 'local', 'RMSE_medio': 0.12641394381717022, 'RMSE_dp': 0.0003881161394177919, 'MAE_medio': 0.07058176447530998, 'sementes': 5}
Incluindo persistência: rede


,arquitetura,lag,metodo,RMSE_medio,RMSE_dp,MAE_medio,sementes
6,lstm_direta,12,local,0.12641,0.00039,0.07058,5
5,lstm_direta,12,completo,0.12653,0.00020,0.07095,5
13,mlp_direta,12,completo,0.12676,0.00030,0.06991,5
22,mlp_residual,12,local,0.12682,0.00026,0.06986,5
2,lstm_direta,6,local,0.12690,0.00029,0.07158,5
18,mlp_residual,6,local,0.12698,0.00024,0.07082,5
9,mlp_direta,6,completo,0.12705,0.00025,0.07123,5
1,lstm_direta,6,completo,0.12727,0.00013,0.07248,5
10,mlp_direta,6,local,0.12731,0.00046,0.07201,5
21,mlp_residual,12,completo,0.12737,0.00011,0.07142,5


,arquitetura,metodo,parametros_treinaveis
0,lstm_direta,cabeca,17
1,lstm_direta,completo,1489
2,lstm_direta,local,1489
3,lstm_direta,sem_ajuste,0
4,mlp_direta,cabeca,17
5,mlp_direta,completo,1537
6,mlp_direta,local,1537
7,mlp_direta,sem_ajuste,0
8,mlp_residual,cabeca,17
9,mlp_residual,completo,1537


## Reexecutar cálculos

Use o terminal para evitar treinos longos ocultos em uma célula:

```powershell
.venv\Scripts\python.exe experimentos\transfer_learning_australia_brasil\executar.py preparar
.venv\Scripts\python.exe experimentos\transfer_learning_australia_brasil\executar.py pretreinar
.venv\Scripts\python.exe experimentos\transfer_learning_australia_brasil\executar.py adaptar
.venv\Scripts\python.exe experimentos\transfer_learning_australia_brasil\executar.py avaliar
```

Históricos de 30 e 60 minutos (6/12 passos). São cinco sementes, MLP direta, MLP residual e LSTM direta. Treino fonte: uma de cada 12 janelas diurnas válidas, por custo de CPU, mantendo 2013–2022; validação 2023 e teste 2024–2025 em todos os instantes elegíveis. Treino brasileiro até setembro, validação outubro, teste novembro–dezembro de 2025. Os dados são autocorrelacionados: número de instantes não equivale a observações independentes. A configuração fixa máximos de 25 épocas fonte e 40 na adaptação; são experimentos iniciais, sem busca no teste.
